In [28]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

model = "llama-3.3-70b-versatile"

messages = [
    {
        "role": "system",
        "content": "You are a helpful personal assistant. Use your tools when you need real data."
    }
]

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_calendar",
            "description": "Check calendar events for a given date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string"
                    }
                },
                "required": ["date"]
            }
        }
    }
]


def check_calendar(date):
    return "10am: Standup, 2pm: Dentist"

In [ ]:
messages.append({
    "role": "user",
    "content": "What's on my calendar today?"
})

In [ ]:
while True:

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )

    finish_reason = response.choices[0].finish_reason
    msg = response.choices[0].message

    # Add the LLM's response to conversation history
    messages.append(msg)

    # If the LLM has a final answer, print it and stop
    if finish_reason == "stop":
        print(msg.content)
        break

    # If the LLM wants to use a tool
    if finish_reason == "tool_calls":

        for tc in msg.tool_calls:

            name = tc.function.name
            args = json.loads(tc.function.arguments)

            if name == "check_calendar":
                result = check_calendar(**args)
            else:
                result = f"Unknown tool: {name}"

            # Add tool result to conversation history
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })

Q: What is an AI agent?
A: An AI agent is a computer program that uses artificial intelligence to perform tasks, make decisions, and interact with its environment, often autonomously.

Q: How is that different from a chatbot?
A: An AI agent can perform various tasks and interact with systems, whereas a chatbot is a type of AI agent that primarily interacts with humans through text or voice conversations.

Q: Give me one example.
A: Virtual assistants like Siri or Alexa are examples of AI agents that can perform tasks, while also interacting with humans like a chatbot.

